In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:

import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

c:\Users\audrb\miniconda3\envs\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pdf_loader = PyPDFLoader("./data/transformer.pdf")
pdf_docs = pdf_loader.load()

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 25587.00it/s]


In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_output_tokens=300,
)

In [6]:
test_questions = [
    "Transformer의 핵심 아이디어는 무엇인가요?",
    "Self-Attention이란 무엇인가요?",
    "Multi-Head Attention을 사용하는 이유는 무엇인가요?",
    "Positional Encoding을 사용하는 이유는 무엇인가요?",
    "Encoder와 Decoder의 차이는 무엇인가요?",
    "Scaled Dot-Product Attention은 어떻게 계산되나요?",
    "Transformer가 RNN보다 병렬화에 유리한 이유는 무엇인가요?",
    "Transformer Base 모델에서 attention head는 몇 개인가요?",
    "Transformer의 Encoder는 몇 개의 layer로 구성되어 있나요?",
    "Transformer와 관련 없는 질문에 대해 시스템은 어떻게 응답해야 하나요?"
]

In [7]:
eval_cases = [
    {
        "question": "Transformer의 핵심 아이디어는 무엇인가요?",
        "retrieval_keywords": [
            ["attention"],
            ["recurrence", "순환"],
            ["convolution", "합성곱"]
        ],
        "answer_keywords": [
            ["attention", "어텐션"],
            ["recurrence", "순환"],
            ["convolution", "합성곱"]
        ]
    },

    {
        "question": "Self-Attention이란 무엇인가요?",
        "retrieval_keywords": [
            ["self-attention", "self attention"],
            ["sequence", "시퀀스"],
            ["positions", "위치"]
        ],
        "answer_keywords": [
            ["self-attention", "self attention"],
            ["sequence", "시퀀스"],
            ["positions", "위치"]
        ]
    },

    {
        "question": "Multi-Head Attention을 사용하는 이유는 무엇인가요?",
        "retrieval_keywords": [
            ["multi-head"],
            ["representation subspaces"],
            ["positions"]
        ],
        "answer_keywords": [
            ["multi-head"],
            ["representation subspaces"],
            ["positions"]
        ]
    },

    {
        "question": "Positional Encoding을 사용하는 이유는 무엇인가요?",
        "retrieval_keywords": [
            ["positional encoding"],
            ["position", "positions"],
            ["recurrence", "convolution"]
        ],
        "answer_keywords": [
            ["positional encoding"],
            ["position", "위치"],
            ["recurrence", "convolution"]
        ]
    },

    {
        "question": "Encoder와 Decoder의 차이는 무엇인가요?",
        "retrieval_keywords": [
            ["encoder"],
            ["decoder"],
            ["multi-head attention"]
        ],
        "answer_keywords": [
            ["encoder"],
            ["decoder"],
            ["third", "세 번째", "three"]
        ]
    },

    {
        "question": "Scaled Dot-Product Attention은 어떻게 계산되나요?",
        "retrieval_keywords": [
            ["scaled dot-product attention"],
            ["softmax"],
            ["Q", "K", "V"]
        ],
        "answer_keywords": [
            ["scaled dot-product attention"],
            ["softmax"],
            ["Q", "K", "V"]
        ]
    },

    {
        "question": "Transformer가 RNN보다 병렬화에 유리한 이유는 무엇인가요?",
        "retrieval_keywords": [
            ["parallel"],
            ["recurrent", "recurrence"],
            ["sequential"]
        ],
        "answer_keywords": [
            ["parallel", "병렬"],
            ["recurrent", "순환"],
            ["sequential", "순차"]
        ]
    },

    {
        "question": "Transformer Base 모델에서 attention head는 몇 개인가요?",
        "retrieval_keywords": [
            ["8"],
            ["head", "heads"]
        ],
        "answer_keywords": [
            ["8"],
            ["head", "heads"]
        ]
    },

    {
        "question": "Transformer의 Encoder는 몇 개의 layer로 구성되어 있나요?",
        "retrieval_keywords": [
            ["6"],
            ["encoder"],
            ["layer"]
        ],
        "answer_keywords": [
            ["6"],
            ["encoder"],
            ["layer"]
        ]
    },

    {
        "question": "Transformer와 관련 없는 질문에 대해 시스템은 어떻게 응답해야 하나요?",
        "retrieval_keywords": [],
        "answer_keywords": [
            ["주어진 정보로는 답변할 수 없습니다"]
        ]
    }
]

In [8]:
def check_keyword_groups(text, keyword_groups):

    text = text.lower()

    matched_groups = 0

    for group in keyword_groups:

        if any(
            keyword.lower() in text
            for keyword in group
        ):
            matched_groups += 1

    return matched_groups

In [9]:
experiment_results = []

In [10]:
chunk_sizes = [300, 500, 800, 1000]
overlaps = [0, 50, 100, 150]

In [11]:
for chunk_size in chunk_sizes:

    for chunk_overlap in overlaps:

        print(
            f"실험 중: chunk_size={chunk_size}, "
            f"overlap={chunk_overlap}"
        )

        # -----------------------------
        # 1. Chunking
        # -----------------------------

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n"]
        )

        texts = text_splitter.split_documents(pdf_docs)


        # -----------------------------
        # 2. Chroma 생성
        # -----------------------------

        vectorstore = Chroma.from_documents(
            documents=texts,
            embedding=embeddings,
            collection_name=f"experiment_{chunk_size}_{chunk_overlap}",
        )


        # -----------------------------
        # 3. Retriever
        # -----------------------------

        retriever = vectorstore.as_retriever(
            search_kwargs={"k": 2}
        )


        # -----------------------------
        # 4. Prompt
        # -----------------------------

        prompt = ChatPromptTemplate.from_template("""
        다음 컨텍스트를 바탕으로 질문에 답변해주세요.

        컨텍스트에 관련 정보가 없다면
        "주어진 정보로는 답변할 수 없습니다."
        라고 답변해주세요.

        컨텍스트:
        {context}

        질문:
        {input}

        답변:
        """)


        # -----------------------------
        # 5. RAG Chain
        # -----------------------------

        combine_docs_chain = create_stuff_documents_chain(
            llm,
            prompt
        )

        rag_chain = create_retrieval_chain(
            retriever,
            combine_docs_chain
        )


        # -----------------------------
        # 6. 질문별 점수 저장
        # -----------------------------

        retrieval_scores = []
        answer_scores = []


        # -----------------------------
        # 7. 10개 질문 테스트
        # -----------------------------

        for case in eval_cases:

            question = case["question"]

            response = rag_chain.invoke({
                "input": question
            })


            # -------------------------
            # 검색된 Context
            # -------------------------

            context = "\n\n".join(
                doc.page_content
                for doc in response["context"]
            )


            # -------------------------
            # Retrieval 평가
            # -------------------------

            retrieval_keywords = case[
                "retrieval_keywords"
            ]

            if len(retrieval_keywords) == 0:

                retrieval_score = 1.0

            else:

                matched = check_keyword_groups(
                    context,
                    retrieval_keywords
                )

                retrieval_score = (
                    matched /
                    len(retrieval_keywords)
                )


            # -------------------------
            # Answer 평가
            # -------------------------

            answer = response["answer"]

            answer_keywords = case[
                "answer_keywords"
            ]

            matched = check_keyword_groups(
                answer,
                answer_keywords
            )

            answer_score = (
                matched /
                len(answer_keywords)
            )


            retrieval_scores.append(
                retrieval_score
            )

            answer_scores.append(
                answer_score
            )


        # -----------------------------
        # 8. 전체 질문 평균
        # -----------------------------

        retrieval_accuracy = (
            sum(retrieval_scores)
            / len(retrieval_scores)
        )

        answer_accuracy = (
            sum(answer_scores)
            / len(answer_scores)
        )


        # -----------------------------
        # 9. 최종 결과만 저장
        # -----------------------------

        experiment_results.append({

            "chunk_size": chunk_size,

            "chunk_overlap": chunk_overlap,

            "retrieval_accuracy":
                retrieval_accuracy,

            "answer_accuracy":
                answer_accuracy
        })


        print(
            f"Retrieval: "
            f"{retrieval_accuracy * 100:.1f}% | "
            f"Answer: "
            f"{answer_accuracy * 100:.1f}%"
        )

실험 중: chunk_size=300, overlap=0


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Retrieval: 68.3% | Answer: 50.0%
실험 중: chunk_size=300, overlap=50


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Retrieval: 71.7% | Answer: 50.0%
실험 중: chunk_size=300, overlap=100


...

Retrieval: 90.0% | Answer: 83.3%
실험 중: chunk_size=1000, overlap=100


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Retrieval: 93.3% | Answer: 90.0%
실험 중: chunk_size=1000, overlap=150


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieval: 93.3% | Answer: 90.0%


In [12]:
df = pd.DataFrame(experiment_results)

df

,chunk_size,chunk_overlap,retrieval_accuracy,answer_accuracy
0,300,0,0.683333,0.500000
1,300,50,0.716667,0.500000
2,300,100,0.716667,0.533333
3,300,150,0.716667,0.466667
4,500,0,0.766667,0.716667
5,500,50,0.900000,0.750000
6,500,100,0.933333,0.800000
7,500,150,0.800000,0.700000
8,800,0,0.900000,0.833333
9,800,50,0.900000,0.733333


In [13]:
best_result = df.loc[
    df["answer_accuracy"].idxmax()
]

best_result

chunk_size            1000.000000
chunk_overlap          100.000000
retrieval_accuracy       0.933333
answer_accuracy          0.900000
Name: 14, dtype: float64

In [14]:
df.to_csv(
    "rag_chunk_experiment.csv",
    index=False,
    encoding="utf-8-sig"
)

In [16]:
K_value = [1,2,3,4,5]

In [18]:
for K in K_value:

    print(
        f"실험 중: K_value={K}, "
    )


    text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=100,
            length_function=len,
            separators=["\n\n", "\n"]
        )

    texts = text_splitter.split_documents(pdf_docs)


    vectorstore = Chroma.from_documents(
            documents=texts,
            embedding=embeddings,
            collection_name=f"experiment_{K}",
        )



    retriever = vectorstore.as_retriever(
            search_kwargs={"k": K}
        )



    prompt = ChatPromptTemplate.from_template("""
        다음 컨텍스트를 바탕으로 질문에 답변해주세요.

        컨텍스트에 관련 정보가 없다면
        "주어진 정보로는 답변할 수 없습니다."
        라고 답변해주세요.

        컨텍스트:
        {context}

        질문:
        {input}

        답변:
        """)




    combine_docs_chain = create_stuff_documents_chain(
            llm,
            prompt
        )

    rag_chain = create_retrieval_chain(
            retriever,
            combine_docs_chain
        )



    retrieval_scores = []
    answer_scores = []




    for case in eval_cases:

            question = case["question"]

            response = rag_chain.invoke({
                "input": question
            })


            # -------------------------
            # 검색된 Context
            # -------------------------

            context = "\n\n".join(
                doc.page_content
                for doc in response["context"]
            )


            # -------------------------
            # Retrieval 평가
            # -------------------------

            retrieval_keywords = case[
                "retrieval_keywords"
            ]

            if len(retrieval_keywords) == 0:

                retrieval_score = 1.0

            else:

                matched = check_keyword_groups(
                    context,
                    retrieval_keywords
                )

                retrieval_score = (
                    matched /
                    len(retrieval_keywords)
                )



            answer = response["answer"]

            answer_keywords = case[
                "answer_keywords"
            ]

            matched = check_keyword_groups(
                answer,
                answer_keywords
            )

            answer_score = (
                matched /
                len(answer_keywords)
            )


            retrieval_scores.append(
                retrieval_score
            )

            answer_scores.append(
                answer_score
            )



    retrieval_accuracy = (
            sum(retrieval_scores)
            / len(retrieval_scores)
        )

    answer_accuracy = (
            sum(answer_scores)
            / len(answer_scores)
        )



    experiment_results.append({

            "K_value": K,

            "retrieval_accuracy":
                retrieval_accuracy,

            "answer_accuracy":
                answer_accuracy
        })


    print(
            f"K_value:"
            f"{K}\n\n"
            f"Retrieval: "
            f"{retrieval_accuracy * 100:.1f}% | "
            f"Answer: "
            f"{answer_accuracy * 100:.1f}%"
        )

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


실험 중: K_value=1, 


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


K_value:1

Retrieval: 86.7% | Answer: 73.3%
실험 중: K_value=2, 


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


...

K_value:3

Retrieval: 93.3% | Answer: 86.7%
실험 중: K_value=4, 


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


K_value:4

Retrieval: 93.3% | Answer: 90.0%
실험 중: K_value=5, 


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


K_value:5

Retrieval: 93.3% | Answer: 90.0%


In [19]:
experiment_results


[{'chunk_size': 300,
  'chunk_overlap': 0,
  'retrieval_accuracy': 0.6833333333333333,
  'answer_accuracy': 0.5},
 {'chunk_size': 300,
  'chunk_overlap': 50,
  'retrieval_accuracy': 0.7166666666666666,
  'answer_accuracy': 0.5},
 {'chunk_size': 300,
  'chunk_overlap': 100,
  'retrieval_accuracy': 0.7166666666666666,
  'answer_accuracy': 0.5333333333333334},
 {'chunk_size': 300,
  'chunk_overlap': 150,
  'retrieval_accuracy': 0.7166666666666666,
  'answer_accuracy': 0.4666666666666666},
 {'chunk_size': 500,
  'chunk_overlap': 0,
  'retrieval_accuracy': 0.7666666666666666,
  'answer_accuracy': 0.7166666666666666},
 {'chunk_size': 500,
  'chunk_overlap': 50,
  'retrieval_accuracy': 0.9,
  'answer_accuracy': 0.75},
 {'chunk_size': 500,
  'chunk_overlap': 100,
  'retrieval_accuracy': 0.9333333333333332,
  'answer_accuracy': 0.8},
 {'chunk_size': 500,
  'chunk_overlap': 150,
  'retrieval_accuracy': 0.8,
  'answer_accuracy': 0.7},
 {'chunk_size': 800,
  'chunk_overlap': 0,
  'retrieval_accura

In [34]:
df = pd.DataFrame(experiment_results)

df

,chunk_size,chunk_overlap,retrieval_accuracy,answer_accuracy,K_value
0,300.0,0.0,0.683333,0.500000,NaN
1,300.0,50.0,0.716667,0.500000,NaN
2,300.0,100.0,0.716667,0.533333,NaN
3,300.0,150.0,0.716667,0.466667,NaN
4,500.0,0.0,0.766667,0.716667,NaN
5,500.0,50.0,0.900000,0.750000,NaN
6,500.0,100.0,0.933333,0.800000,NaN
7,500.0,150.0,0.800000,0.700000,NaN
8,800.0,0.0,0.900000,0.833333,NaN
9,800.0,50.0,0.900000,0.733333,NaN


In [35]:
K_best_result = df.loc[
    df[16:]["answer_accuracy"].idxmax()
]

K_best_result

chunk_size                 NaN
chunk_overlap              NaN
retrieval_accuracy    0.933333
answer_accuracy       0.900000
K_value               2.000000
Name: 17, dtype: float64

In [36]:
df.to_csv(
    "rag_Chunk_K_experiment.csv",
    index=False,
    encoding="utf-8-sig"
)